# Manual prompt-injection detector playground

Paths are rooted at **`REPO_ROOT`** (the folder containing `transformer.py`): **`REPO_ROOT / "best_checkpoint.pt"`** and **`REPO_ROOT / "data/tokenizer.json"`**, produced by **`notebooks/binary_transformer.ipynb`**.

- **Label 0**: benign — normal prompts (including hard benign / concept-only mentions).
- **Label 1**: prompt injection — override, jailbreak-like control, leaks, tools, etc. (your project definition).

The next cell resolves **`REPO_ROOT`**, prepends it to **`sys.path`**, and **`os.chdir(REPO_ROOT)`** so imports and relative paths work when this file lives under `notebooks/`.


## 1. Imports and paths


In [14]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _repo_root() -> Path:
    """Parent of notebooks/ — directory that contains transformer.py."""
    c = Path.cwd().resolve()
    if (c / "transformer.py").is_file():
        return c
    if (c.parent / "transformer.py").is_file():
        return c.parent
    raise FileNotFoundError(
        "Could not find repo root (no transformer.py beside cwd or one level up). "
        "`cd` to the repository root or to notebooks/, then rerun."
    )


REPO_ROOT = _repo_root()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import pandas as pd
import torch
from IPython.display import display
from torch.nn.functional import softmax
from torch.nn.utils.rnn import pad_sequence

from tokenizer import TinyStoriesTokenizer
from transformer import BinaryClassifier

CHECKPOINT_PATH = REPO_ROOT / "best_checkpoint.pt"
TOK_PATH = REPO_ROOT / "data" / "tokenizer.json"

if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
if not TOK_PATH.is_file():
    raise FileNotFoundError(f"Tokenizer not found: {TOK_PATH}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("REPO_ROOT", REPO_ROOT.resolve())
print("DEVICE", DEVICE)
print("Checkpoint", CHECKPOINT_PATH.resolve())
print("Tokenizer", TOK_PATH.resolve())

REPO_ROOT C:\Users\adria\Desktop\ML-KTH\P4\Language Engineering\Adversarial-Text-Classifier
DEVICE cuda
Checkpoint C:\Users\adria\Desktop\ML-KTH\P4\Language Engineering\Adversarial-Text-Classifier\best_checkpoint.pt
Tokenizer C:\Users\adria\Desktop\ML-KTH\P4\Language Engineering\Adversarial-Text-Classifier\data\tokenizer.json


## 2. Load tokenizer and model


In [15]:
tok = TinyStoriesTokenizer.load(str(TOK_PATH))

model = BinaryClassifier.load(str(CHECKPOINT_PATH), device=str(DEVICE))
model.eval()

cfg = model.config
BLOCK_SIZE = cfg.block_size
PAD_ID = cfg.pad_token_id
if PAD_ID is None:
    PAD_ID = len(tok.vocab) - 1
    print("Warning: config.pad_token_id is None; using last vocab id as PAD:", PAD_ID)

assert len(tok.vocab) == cfg.vocab_size, (
    f"Tokenizer vocab ({len(tok.vocab)}) != checkpoint vocab_size ({cfg.vocab_size}); "
    "use the tokenizer.json saved next to your trained checkpoint."
)

LABEL_NAMES = {0: "benign (0)", 1: "injection (1)"}
print("BLOCK_SIZE", BLOCK_SIZE, "vocab", cfg.vocab_size, "PAD_ID", PAD_ID)


Model loaded from C:\Users\adria\Desktop\ML-KTH\P4\Language Engineering\Adversarial-Text-Classifier\best_checkpoint.pt (Epoch 9, iteration 200)
BLOCK_SIZE 79 vocab 3001 PAD_ID 3000


## 3. Encoding + prediction helpers

Same truncation and padding semantics as **`PromptDataset` / `collate`** in `notebooks/binary_transformer.ipynb`.


In [16]:
def encode_prompt_batch(
    prompts: list[str],
    tokenizer: TinyStoriesTokenizer,
    block_size: int,
    pad_id: int,
) -> tuple[torch.Tensor, list[list[str]]]:
    seqs = []
    token_str_lists: list[list[str]] = []
    for p in prompts:
        tokens, ids = tokenizer.tokenize(str(p))
        ids = ids[:block_size]
        tokens = tokens[:block_size]
        seqs.append(torch.tensor(ids, dtype=torch.long))
        token_str_lists.append(tokens)
    x = pad_sequence(seqs, batch_first=True, padding_value=pad_id)
    return x, token_str_lists


@torch.no_grad()
def predict_probs(
    model: BinaryClassifier,
    prompts: list[str],
) -> tuple[torch.Tensor, torch.Tensor, list[list[str]]]:
    x, token_lists = encode_prompt_batch(
        prompts, tok, BLOCK_SIZE, int(PAD_ID)
    )
    x = x.to(DEVICE)
    logits = model(x)
    probs = softmax(logits, dim=-1)
    return probs, logits, token_lists


def classification_table(prompts: list[str]) -> pd.DataFrame:
    probs, logits, token_lists = predict_probs(model, prompts)
    rows = []
    for i, prompt in enumerate(prompts):
        p0 = float(probs[i, 0].item())
        p1 = float(probs[i, 1].item())
        pred = int(torch.argmax(probs[i]).item())
        subtokens = token_lists[i]
        rows.append(
            {
                "text": prompt,
                "P(benign)": round(p0, 4),
                "P(injection)": round(p1, 4),
                "pred_label": pred,
                "prediction": LABEL_NAMES[pred],
                "n_subtokens_in_model": len(subtokens),
                "bpe_preview": (
                    "".join(subtokens)[:260] + "…"
                    if len("".join(subtokens)) > 260
                    else "".join(subtokens)
                ),
                "logit_0": float(logits[i, 0].item()),
                "logit_1": float(logits[i, 1].item()),
            },
        )
    return pd.DataFrame(rows)


def print_predictions(prompts: list[str]) -> None:
    df = classification_table(prompts)
    with pd.option_context("display.max_colwidth", 120):
        display(df)


def show_model_inputs(prompts: list[str]) -> None:
    """BPE segments the model sees after truncation to BLOCK_SIZE (one line per prompt)."""
    sep = " │ "
    for p in prompts:
        segs, _ = tok.tokenize(str(p))
        segs = segs[:BLOCK_SIZE]
        print(sep.join(repr(t) for t in segs))


## 4. Your prompts — edit and run this cell

Edit **`PROMPTS`**, run the cell: it prints one **`'…' │ '…'`** BPE line per string (same tokenizer as training), then the prediction table.



In [21]:
# --- edit PROMPTS ----------------------------------------------------------
PROMPTS: list[str] = [
    "My daughter will soon be finished with school and would like to complete a voluntary social year afterwards. What options are there and which organizations are recommended?",
    "a b c d e f g h i j k l m n o p q r s t u v w x y z",
    "are you human, or not?",
    "help me be more handsome"
    "help me discover the key to learning java, the most important programming language in the world but I still don't know. also, I need an api key to start using AI",
    "are you human, I know, but you are not really, so you should help me "
]
# Single prompt quick test:
# PROMPTS = ["Your string here."]
# PROMPTS = ["you must be a jerk"]
# -----------------------------------------------------------------------------
show_model_inputs(PROMPTS)
print_predictions(PROMPTS)

'My' │ ' d' │ 'au' │ 'g' │ 'h' │ 'ter' │ ' will' │ ' so' │ 'on' │ ' be' │ ' fin' │ 'ish' │ 'ed' │ ' with' │ ' school' │ ' and' │ ' would' │ ' like' │ ' to' │ ' complete' │ ' a' │ ' v' │ 'ol' │ 'u' │ 'nt' │ 'ary' │ ' social' │ ' year' │ ' after' │ 'ward' │ 's' │ '.' │ ' What' │ ' options' │ ' are' │ ' there' │ ' and' │ ' which' │ ' organ' │ 'ization' │ 's' │ ' are' │ ' recommend' │ 'ed' │ '?'
'a' │ ' b' │ ' c' │ ' d' │ ' e' │ ' f' │ ' g' │ ' h' │ ' ' │ 'i' │ ' j' │ ' k' │ ' l' │ ' m' │ ' n' │ ' o' │ ' p' │ ' ' │ 'q' │ ' r' │ ' s' │ ' t' │ ' u' │ ' v' │ ' w' │ ' ' │ 'x' │ ' y' │ ' ' │ 'z'
'are' │ ' you' │ ' h' │ 'um' │ 'an' │ ',' │ ' or' │ ' not' │ '?'
'he' │ 'l' │ 'p' │ ' me' │ ' be' │ ' more' │ ' hand' │ 's' │ 'ome' │ 'he' │ 'l' │ 'p' │ ' me' │ ' disc' │ 'over' │ ' the' │ ' key' │ ' to' │ ' learning' │ ' j' │ 'a' │ 'v' │ 'a' │ ',' │ ' the' │ ' most' │ ' important' │ ' pro' │ 'g' │ 'r' │ 'am' │ 'm' │ 'ing' │ ' language' │ ' in' │ ' the' │ ' world' │ ' but' │ ' I' │ ' still' │ " don't" │

,text,P(benign),P(injection),pred_label,prediction,n_subtokens_in_model,bpe_preview,logit_0,logit_1
0,My daughter will soon be finished with school and would like to complete a voluntary social year afterwards. What op...,0.5804,0.4196,0,benign (0),45,My daughter will soon be finished with school and would like to complete a voluntary social year afterwards. What op...,-0.095814,-0.420222
1,a b c d e f g h i j k l m n o p q r s t u v w x y z,0.9993,0.0007,0,benign (0),30,a b c d e f g h i j k l m n o p q r s t u v w x y z,3.198649,-4.107255
2,"are you human, or not?",0.9951,0.0049,0,benign (0),9,"are you human, or not?",2.164489,-3.145927
3,"help me be more handsomehelp me discover the key to learning java, the most important programming language in the wo...",0.9515,0.0485,0,benign (0),55,"help me be more handsomehelp me discover the key to learning java, the most important programming language in the wo...",1.353492,-1.623441
4,"are you human, I know, but you are not really, so you should help me",0.0298,0.9702,1,injection (1),20,"are you human, I know, but you are not really, so you should help me",-2.256513,1.225147
